# Deep Learning 015 — Backpropagation, Part 1: The "What"

Companion notebook to the lesson. Backpropagation is not a new idea on top of gradient
descent — it is gradient descent, plus an efficient order for computing the derivatives.

The network from the lesson: **2 inputs → 1 hidden layer of 2 nodes → 1 output**, all
linear, trained on one example at a time with MSE.

That is **9 parameters**, so every step needs 9 derivatives. We compute all of them by
hand, then check every single one against a numerical derivative.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# 9 parameters: W1 (2x2) + b1 (2) + W2 (2x1) + b2 (1)
def init():
    return dict(W1=rng.normal(size=(2, 2)) * 0.5, b1=np.zeros(2),
                W2=rng.normal(size=(2, 1)) * 0.5, b2=np.zeros(1))

def forward(p, x):
    o1 = x @ p["W1"] + p["b1"]        # hidden layer, linear
    y_hat = o1 @ p["W2"] + p["b2"]    # output, linear
    return o1, float(y_hat[0])

def loss(p, x, y):
    return (y - forward(p, x)[1]) ** 2

params = init()
x = np.array([0.8, -0.3])
y = 1.5
o1, y_hat = forward(params, x)
print(f"hidden activations {np.round(o1, 4)}   prediction {y_hat:.4f}   loss {loss(params, x, y):.4f}")
print(f"parameter count: {sum(v.size for v in params.values())}")

## The four steps

1. Pick an example, run it **forward**, get $\hat y$.
2. Compute the loss $L = (y - \hat y)^2$.
3. Compute $\partial L / \partial w$ for **every** parameter.
4. Update every parameter: $w \leftarrow w - \eta \,\partial L/\partial w$.

Step 3 is the only hard one, and it is where the name comes from: the derivatives are
computed **from the output backwards**, because the output-layer derivatives are the
ingredients of the hidden-layer ones.

## Doing step 3 by hand

Write $L = (y - \hat y)^2$ with $\hat y = o_{11}w_{2,1} + o_{12}w_{2,2} + b_2$.

Everything starts from one quantity:

$$\frac{\partial L}{\partial \hat y} = -2(y - \hat y)$$

and the chain rule multiplies it by how $\hat y$ depends on each parameter. **Every one of
the nine derivatives reuses this same number.**

In [ ]:
def backward(p, x, y):
    o1, y_hat = forward(p, x)
    dL_dyhat = -2 * (y - y_hat)                     # <- computed ONCE, reused nine times

    # output layer: y_hat = o1 . W2 + b2
    dW2 = dL_dyhat * o1.reshape(-1, 1)              # d y_hat / d W2 = o1
    db2 = np.array([dL_dyhat * 1.0])                # d y_hat / d b2 = 1

    # hidden layer: the signal has to travel back through W2 first
    dL_do1 = dL_dyhat * p["W2"].ravel()             # d y_hat / d o1 = W2
    dW1 = np.outer(x, dL_do1)                       # d o1 / d W1 = x
    db1 = dL_do1 * 1.0                              # d o1 / d b1 = 1
    return dict(W1=dW1, b1=db1, W2=dW2, b2=db2)

grads = backward(params, x, y)
for k, v in grads.items():
    print(f"  dL/d{k}: {np.round(v.ravel(), 5)}")

## Check every one of them numerically

An analytical gradient you have not checked is a guess. The definition of a derivative is

$$\frac{\partial L}{\partial w} \approx \frac{L(w + h) - L(w - h)}{2h}$$

so nudge each parameter and see whether the loss moves the way the formula says.

In [ ]:
def numeric_grads(p, x, y, h=1e-6):
    out = {}
    for k, arr in p.items():
        g = np.zeros_like(arr, dtype=float)
        it = np.nditer(arr, flags=["multi_index"])
        while not it.finished:
            i = it.multi_index
            old = arr[i]
            arr[i] = old + h; up = loss(p, x, y)
            arr[i] = old - h; dn = loss(p, x, y)
            arr[i] = old
            g[i] = (up - dn) / (2 * h)
            it.iternext()
        out[k] = g
    return out

num = numeric_grads(params, x, y)
worst = 0.0
for k in params:
    err = np.abs(num[k] - grads[k]).max()
    worst = max(worst, err)
    print(f"  {k:>3}: analytic {np.round(grads[k].ravel(), 5)}"
          f"   numeric {np.round(num[k].ravel(), 5)}   max err {err:.2e}")
assert worst < 1e-6
print(f"\nall 9 derivatives agree, worst error {worst:.1e}")

All nine match. The hand-derived formulas are the real thing, not an approximation —
the numerical version is the slow check, not the definition.

**Why not just use the numerical version?** Count the work: it needs *two forward passes
per parameter*. Nine parameters, eighteen forward passes. Backpropagation gets all nine
from **one** forward pass and one backward pass.

In [ ]:
n_params = sum(v.size for v in params.values())
print(f"numerical: {2 * n_params} forward passes for {n_params} parameters")
print(f"backprop : 1 forward + 1 backward, regardless of the parameter count")
print(f"\nfor a network with 25 million parameters that is "
      f"{2 * 25_000_000:,} forward passes per step, against 1")

## Now the whole training loop

Steps 1–4, repeated. Watch the loss fall on a single example first — the simplest possible
demonstration that the derivatives point the right way.

In [ ]:
params = init()
lr = 0.01
print(f"{'epoch':>7}{'loss':>12}{'prediction':>14}")
for epoch in range(0, 201):
    g = backward(params, x, y)
    if epoch % 40 == 0:
        print(f"{epoch:>7}{loss(params, x, y):>12.6f}{forward(params, x)[1]:>14.6f}")
    for k in params:
        params[k] -= lr * g[k]
print(f"\ntarget was {y}")

## And on a real dataset

The same nine derivatives, cycling through examples. The data is generated from a known
linear rule so we can check the network recovers it.

In [ ]:
X = rng.normal(size=(400, 2))
true_w, true_b = np.array([2.0, -3.0]), 0.5
Y = X @ true_w + true_b + rng.normal(scale=0.1, size=400)

params = init()
lr = 0.005
full_mse = lambda: np.mean([loss(params, X[i], Y[i]) for i in range(len(X))])
print(f"before training   MSE {full_mse():.6f}")
for epoch in range(60):
    order = rng.permutation(len(X))
    for i in order:
        g = backward(params, X[i], Y[i])
        for k in params:
            params[k] -= lr * g[k]
    if epoch % 15 == 0 or epoch == 59:
        print(f"epoch {epoch:>3}         MSE {full_mse():.6f}")

# the two linear layers collapse to one effective weight vector
eff_w = (params["W1"] @ params["W2"]).ravel()
eff_b = float((params["b1"] @ params["W2"] + params["b2"])[0])
print(f"\nnoise floor is 0.01 by construction (sigma = 0.1), so it is done at epoch 0")
print(f"effective weights {np.round(eff_w, 3)} vs true {true_w}")
print(f"effective bias    {eff_b:.3f} vs true {true_b}")

Note what the last two lines expose. Two **linear** layers stacked are algebraically one
linear layer: `W1 @ W2` is just some other 2×1 matrix. The network has nine parameters but
only three degrees of freedom, and it recovers exactly the rule the data came from.

That is not a flaw in backpropagation — it is a fact about linear networks, and it is the
reason activation functions exist. Lesson 016 does the same derivation with a sigmoid in
the middle, where the layers no longer collapse.

## Try it yourself

1. Change `lr` to 0.5 in the single-example loop. What happens, and at what value does it
   stop converging?
2. Add a third hidden node. How many parameters now, and how many numerical forward passes
   would the gradient cost?
3. Corrupt one of the returned gradients (multiply `dW1` by 2) and re-run the numeric
   check. Confirm the check catches it — this is how you debug a hand-written backward pass.
4. Print `dL_dyhat` at epoch 0 and epoch 200 of the single-example loop. Why does it shrink,
   and what does that imply about the size of the updates near a minimum?